In [2]:
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv('sample_data/diabetes_prediction_dataset.csv')
df.head(10)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0
5,Female,20.0,0,0,never,27.32,6.6,85,0
6,Female,44.0,0,0,never,19.31,6.5,200,1
7,Female,79.0,0,0,No Info,23.86,5.7,85,0
8,Male,42.0,0,0,never,33.64,4.8,145,0
9,Female,32.0,0,0,never,27.32,5.0,100,0


In [9]:
# Data quality checks
# missing values, duplicates
print (f"Missing values: {df.isnull().sum()}")
print (f"Duplicates: {df.duplicated().sum()}")
print (f"Target distribution: {df['diabetes'].value_counts()}")
print (f"Target balance: {(df['diabetes'].value_counts(normalize=True) * 100).round(2).to_dict()}")
df.shape

Missing values: gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64
Duplicates: 3854
Target distribution: diabetes
0    91500
1     8500
Name: count, dtype: int64
Target balance: {0: 91.5, 1: 8.5}


(100000, 9)

In [14]:
# Statistical distribution
print(df.describe())
print(df['gender'].unique())
print(df['smoking_history'].unique())

                 age  hypertension  heart_disease            bmi  \
count  100000.000000  100000.00000  100000.000000  100000.000000   
mean       41.885856       0.07485       0.039420      27.320767   
std        22.516840       0.26315       0.194593       6.636783   
min         0.080000       0.00000       0.000000      10.010000   
25%        24.000000       0.00000       0.000000      23.630000   
50%        43.000000       0.00000       0.000000      27.320000   
75%        60.000000       0.00000       0.000000      29.580000   
max        80.000000       1.00000       1.000000      95.690000   

         HbA1c_level  blood_glucose_level       diabetes  
count  100000.000000        100000.000000  100000.000000  
mean        5.527507           138.058060       0.085000  
std         1.070672            40.708136       0.278883  
min         3.500000            80.000000       0.000000  
25%         4.800000           100.000000       0.000000  
50%         5.800000           14

In [ ]:
print("Feature cardinality test begins \n" + "="*60)
for columns in df.columns:
    num_distinct = len(df[columns].unique())
    feature_type = "Categorical" if num_distinct < 10 else "Numerical"
    print (f"{columns:20s} | {num_distinct:6,} unique values | {feature_type}")


Feature cardinality test begins 
gender               |      3 unique values | Categorical
age                  |    102 unique values | Numerical
hypertension         |      2 unique values | Categorical
heart_disease        |      2 unique values | Categorical
smoking_history      |      6 unique values | Categorical
bmi                  |  4,247 unique values | Numerical
HbA1c_level          |     18 unique values | Numerical
blood_glucose_level  |     18 unique values | Numerical
diabetes             |      2 unique values | Categorical


In [ ]:
# Univariate analysis
import matplotlib.pyplot as plt
import seaborn as sns
fig,axes = plt.subplots(2,2,figsize=(14, 10))
num_features = ['age', 'HbA1c_level', 'bmi', 'blood_glucose_level']
colors = ['blue', 'orange', 'green', 'red']
for idx, (feature, color) in enumerate(zip(num_features, colors)):
    ax = axes[idx//2, idx%2]
    # Histogram with KDE
    ax.hist(df[feature], bins=40, color=color, edgecolor='black', density=True)
    df[feature].plot(kind='kde', ax=ax, color='darkred',linewidth=2)
    # Styling
    ax.set_title(f'Distribution of {feature.replace("_", " ").title()}', fontsize=14)
    ax.set_xlabel(feature.replace('_', ' ').title(), fontsize=12) 
    ax.set_ylabel('Density', fontsize=12)
    ax.grid(alpha=0.3)
    # add statistics
    mean_val = df[feature].mean()
    median_val = df[feature].median()
    ax.axvline(mean_val, color='black', linestyle='--', label=f'Mean: {mean_val:.2f}')
    ax.axvline(median_val, color='gray', linestyle='-.', label=f'Median: {median_val:.2f}')
    ax.legend()
plt.tight_layout()
plt.show()

# Gender distribution
gender_counts = df['gender'].value_counts()
axes[0].bar(gender_counts.index, gender_counts.values, color=['blue', 'orange'])
axes[0].set_title('Gender Distribution', fontsize=14)
axes[0].set_xlabel('Gender', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
for i,v in enumerate(gender_counts.values):
    #axes[0].text(i, v + 500, str(v), ha='center', fontsize=10)
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)

# Smoking history distribution
smoking_counts = df['smoking_history'].value_counts()
axes[1].bar(smoking_counts.index, smoking_counts.values, color=['green', 'red', 'purple', 'cyan', 'magenta'])
axes[1].set_title('Smoking History Distribution', fontsize=14)
axes[1].set_xlabel('Smoking History', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
for i,v in enumerate(smoking_counts.values):
    axes[1].text(i ,v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Bivariate analysis: Age and BMI vs Diabetes
plt.figure(figsize=(14, 6))
ax = sns.scatterplot(data=df, x='age', y='bmi', hue='diabetes', palette={0: '#2ecc71', 1: '#e74c3c'}, alpha=0.6, s=50)
#ax = sns.histplot(data=df, x='age', y='bmi', hue='diabetes', palette='Set1', alpha=0.6, bins=30)
#ax.set_title('Age vs BMI Colored by Diabetes Status', fontsize=14)
#ax = sns.boxplot(data=df, x='age', y='bmi', hue='diabetes', palette='Set2')
plt.title('Age vs BMI Colored by Diabetes Status', fontsize=14, fontweight='bold')
plt.xlabel('Age (years)', fontsize=12)
plt.ylabel('BMI (kg/m²)', fontsize=12)
legend = ax.get_legend()

if legend:
    legend.set_title('Diabetes Status')
    for i, text in enumerate(legend.get_texts()):
        if i == 0: # corresponds to diabetes=0
            text.set_text('Negative')
            text.set_color('#2ecc71')
        elif i == 1: # corresponds to diabetes=1
            text.set_text('Positive')
            text.set_color('#e74c3c')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Clinical markers vs Diabetes status
df['diabetes'] = df['diabetes'].astype('category')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
features = ['bmi', 'age', 'HbA1c_level', 'blood_glucose_level']
titles = ['BMI vs Diabetes Status', 'Age vs Diabetes Status', 'HbA1c Level (%) vs Diabetes Status', 'Blood Glucose Level (mg/dL) vs Diabetes Status']
for idx in range(len(features)):
    feature = features[idx]
    title = titles[idx]
    ax = axes[idx//2, idx%2]
    sns.boxplot(data=df, x='diabetes', y=feature, palette={0: '#2ecc71', 1: '#e74c3c'}, ax=ax) # ax=ax just tells the position where to plot the boxplot
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Diabetes Status', fontsize=12)
    ax.set_ylabel(feature.replace('_', ' ').title(), fontsize=12)
    ax.grid(axis='y', alpha=0.3)

In [ ]:
#Level 3 Data processing and featue engineering
#3.1 Categorical encoding
# one hot encode categorical variable
df_encoded = pd.get_dummies(df, columns=['gender','smoking_history'],drop_first=False)
#3.2 Train test split
df_encoded.dropna(subset=['diabetes'], inplace=True)
x = df_encoded.drop('diabetes', axis=1)
y = df_encoded['diabetes']
x.shape, y.shape



In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# Model training and evaluation

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report, confusion_matrix, precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from Xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
def evaluate_metrics(model, x_train, x_test, y_train, y_test, model_name):
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    y_pred_proba = model.predict_proba(x_test)[:, 1] if hasattr(model, "predict_proba") else None
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    # Cross validation
    cv_scores = cross_val_score(model, x, y, cv=5, scoring='accuracy')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    return f"{model_name} - Accuracy: {accuracy}, F1: {f1}, ROC AUC: {roc_auc}, CV Mean: {cv_mean}, CV Std: {cv_std}, Predictions: {y_pred}, Probabilities: {y_pred_proba}"

models_config = {
    'Logistic Regression': LogisticRegression(C=1, penalty='l2', solver='liblinear', random_state=42, max_iter=200),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'XGBoost': XGBClassifier(n_estimators=100, learning_rate = 0.1, max_depth = 6, random_state=42, use_label_encoder=False, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(n_estimators=200, learning_rate = 0.05, max_depth = -1, random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=300, learning_rate = 0.05, depth = 8, random_state=42, verbose=0)
}

In [ ]:
results = []
predictions_dict = {}
for name, model in models_config.items():
    print(f"Evaluating {name}...", end="")
    result = evaluate_metrics(model, x_train, x_test, y_train, y_test, name)
    results.append(result)
    predictions_dict[name] = {
        'predictions': model.predict(x_test),
        'probabilities': model.predict_proba(x_test)[:, 1] if hasattr(model, "predict_proba") else None
    }
    print(f"Done. {result['model_name']} - Accuracy: {result['accuracy']}, F1: {result['f1']}, ROC AUC: {result['roc_auc']}, CV Mean: {result['cv_mean']}, CV Std: {result['cv_std']}")

In [ ]:
results_df = pd.DataFrame(results)
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
# Metrics comparison
metrics_to_plot = ['accuracy', 'f1', 'roc_auc']
results_melted = results_df.melt(id_vars=['Model'], value_vars=metrics_to_plot, var_name='Metric', value_name='Score')
sns.barplot(data=results_melted, x='Model', y='Score', hue='Metric', ax=axes[0])
axes[0].set_title('Model Performance Comparison', fontsize=14)
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_ylim(0.90, 1.0)
axes[0].legend(title='Metric', loc='lower right')
axes[0].grid(axis='y', alpha=0.3)
plt.setp(axes[0].xaxis.get_xticklabels(), rotation=20, ha='right')

# Cross validation scores
cv_data = results_df[['Model', 'cv_mean', 'cv_std']]
axes[1].barh(cv_data['Model'], cv_data['cv_mean'], xerr=cv_data['cv_std'], color='skyblue', len(cv_data), edgecolor='black', capsize=5)
axes[1].set_title('Cross-Validation Accuracy with Std Dev', fontsize=14)
axes[1].set_xlabel('Accuracy (Mean ± Std)', fontsize=12)
axes[1].set_xlim(0.94, 0.98)
axes[1].grid(axis='x', alpha=0.3)

for i, row in cv_data.iterrows():
    axes[1].text(row['cv_mean'] + 0.005, i, f"{row['cv_mean']:.4f} ± {row['cv_std']:.4f}", va='center', fontsize=10)
plt.tight_layout()
plt.show()
